# Part 1: Probability Distributions, EM for a Gaussian Mixture

**Group Members:** Solomon Leek, Brian Nakuwa, and Ahmad Daib

**Dataset:** Galton family heights. We compare **fathers and children** and treat the heights as one unlabelled pool drawn from a mixture of two Gaussians.

**Requirements:** numpy and pandas. Place the dataset file `GaltonFamilies.csv` in the same folder as this notebook, then run the cells in order. Ensure the filename in the data-loading cell matches the actual CSV filename.

**Model:** p(x) = π₁ · N(x | μ₁, σ₁²) + π₂ · N(x | μ₂, σ₂²)

- **Initialization:** a starting guess for μ₁, μ₂, σ₁², σ₂², π₁, π₂.
- **E-step:** responsibility rᵢₖ = πₖ N(xᵢ | μₖ, σₖ²) / Σⱼ πⱼ N(xᵢ | μⱼ, σⱼ²).
- **M-step:** with Nₖ = Σᵢ rᵢₖ, update μₖ = Σᵢ rᵢₖ xᵢ / Nₖ, σₖ² = Σᵢ rᵢₖ (xᵢ − μₖ)² / Nₖ, πₖ = Nₖ / N.
- **Stop** when the log-likelihood changes by less than a small tolerance.

In [1]:

import numpy as np
import pandas as pd

K = 2

# EM discovers shorter and taller height components, not family roles.
NAMES = ("Shorter-height component", "Taller-height component")

TABLE_COLUMNS = {
    "mu1": "Mean 1 (shorter component)",
    "mu2": "Mean 2 (taller component)",
    "var1": "Variance 1",
    "var2": "Variance 2",
    "pi1": "Mixing weight 1",
    "pi2": "Mixing weight 2",
    "log_likelihood": "Log-Likelihood"
}

## 1. Load the data

The CSV has one row per child, so each father repeats once per child. We keep one row per family for the fathers (205 fathers) so no father is counted twice. Group labels are never used by EM.

In [2]:
df = pd.read_csv("/content/GaltonFamilies.csv")

children = df["childHeight"].to_numpy()
fathers = df.drop_duplicates("family")["father"].to_numpy()
x = np.concatenate([children, fathers])   # unlabelled pool

print(f"Children: {len(children)}  Fathers: {len(fathers)}  Pooled: {len(x)} heights")

Children: 934  Fathers: 205  Pooled: 1139 heights


## 2. EM implementation

Densities and responsibilities are computed in log space so that a test height far from the data cannot underflow to 0/0.

### Initialization and model parameters

We model the pooled heights as a mixture of two Gaussian distributions. Each component has a mean (μₖ), a variance (σₖ²), and a mixing weight (πₖ). The mixing weights show what share of the data belongs to each component, so they have to add up to 1.

To get started, we sort the observed heights and split them into two halves. The mean of each half becomes the starting mean for one component. Both components begin with the same variance, and each gets a mixing weight of 0.5. These are only starting guesses, and the EM algorithm refines them as it works through the data.

In [7]:

def log_gaussian_pdf(x, mu, var):
    """Calculate log N(x | mu, var) from scratch."""
    return -0.5 * np.log(2 * np.pi * var) - (x - mu) ** 2 / (2 * var)


def log_weighted_densities(x, mu, var, pi):
    """Calculate log(pi_k * N(x_i | mu_k, var_k))."""
    return np.column_stack([
        np.log(pi[k]) + log_gaussian_pdf(x, mu[k], var[k])
        for k in range(K)
    ])


def log_likelihood(x, mu, var, pi):
    """Calculate total log-likelihood using the log-sum-exp trick."""
    lw = log_weighted_densities(x, mu, var, pi)
    top = lw.max(axis=1, keepdims=True)

    return np.sum(
        top.ravel() + np.log(np.exp(lw - top).sum(axis=1))
    )


def e_step(x, mu, var, pi):
    """Calculate the responsibility of each component for each point."""
    lw = log_weighted_densities(x, mu, var, pi)
    w = np.exp(lw - lw.max(axis=1, keepdims=True))

    return w / w.sum(axis=1, keepdims=True)


def m_step(x, r):
    """Update component means, variances, and mixing weights."""
    N_k = r.sum(axis=0)

    mu = (r * x[:, None]).sum(axis=0) / N_k
    var = (r * (x[:, None] - mu) ** 2).sum(axis=0) / N_k
    pi = N_k / len(x)

    return mu, var, pi


def initialize(x):
    """Initialize two components without using group labels."""
    s = np.sort(x)
    half = len(s) // 2

    mu = np.array([s[:half].mean(), s[half:].mean()])
    var = np.array([x.var(), x.var()])
    pi = np.array([0.5, 0.5])

    return mu, var, pi


def record(x, iteration, mu, var, pi):
    """Record parameter estimates and log-likelihood."""
    return {
        "iteration": iteration,
        "mu1": mu[0],
        "mu2": mu[1],
        "var1": var[0],
        "var2": var[1],
        "pi1": pi[0],
        "pi2": pi[1],
        "log_likelihood": log_likelihood(x, mu, var, pi)
    }


def run_em(x, max_iter=1000, tol=1e-6):
    """Run EM until log-likelihood changes by less than tol."""
    mu, var, pi = initialize(x)
    history = [record(x, 0, mu, var, pi)]
    converged = False

    for it in range(1, max_iter + 1):
        # E-step: calculate responsibilities.
        r = e_step(x, mu, var, pi)

        # M-step: update model parameters.
        mu, var, pi = m_step(x, r)

        # Record the current iteration.
        history.append(record(x, it, mu, var, pi))

        # Check the change in log-likelihood.
        change = abs(
            history[-1]["log_likelihood"]
            - history[-2]["log_likelihood"]
        )

        if change < tol:
            converged = True
            break

    return mu, var, pi, pd.DataFrame(history), converged


# Run the EM algorithm.
mu, var, pi, history, converged = run_em(
    x, max_iter=1000, tol=1e-6
)

# Report the convergence status accurately.
last_iteration = int(history["iteration"].iloc[-1])
final_log_likelihood = history["log_likelihood"].iloc[-1]

if converged:
    print(f"Converged after {last_iteration} iterations.")
else:
    print(
        f"Stopped after {last_iteration} iterations "
        "without meeting the convergence tolerance."
    )

print(f"Final log-likelihood: {final_log_likelihood:.4f}")

if len(history) >= 2:
    final_change = abs(
        history["log_likelihood"].iloc[-1]
        - history["log_likelihood"].iloc[-2]
    )
    print(f"Final absolute log-likelihood change: {final_change:.8f}")


Converged after 314 iterations.
Final log-likelihood: -3046.9895
Final absolute log-likelihood change: 0.00000098


## 3. Tracking table (initial state and first two iterations)

In [8]:
table = history.head(3).rename(columns=TABLE_COLUMNS)
print(table.round(4).to_string(index=False))

 iteration  Mean 1 (shorter component)  Mean 2 (taller component)  Variance 1  Variance 2  Mixing weight 1  Mixing weight 2  Log-Likelihood
         0                     64.2299                    70.1819     12.7241     12.7241           0.5000           0.5000      -3136.4676
         1                     65.1207                    69.2760      8.7032      8.1149           0.4976           0.5024      -3056.6103
         2                     65.0591                    69.3312      8.5547      7.7733           0.4969           0.5031      -3055.5576


## 4. Classifying a test height

The posterior probabilities for a test height are calculated using the E-step. They indicate how strongly the fitted model associates that height with each Gaussian component.

Set `test_height` to the value provided for the demonstration, in inches.

Component 1 is the shorter-height component, and Component 2 is the taller-height component. These are statistical components learned from the pooled, unlabelled data; they are not guaranteed to correspond exactly to children and fathers because their height distributions overlap.

The two posterior probabilities should sum to approximately 1.

In [5]:
def classify(height, mu, var, pi):
    """Posterior probabilities for one height."""
    p = e_step(np.array([height]), mu, var, pi)[0]
    print(f"Height = {height} inches")
    for name, prob in zip(NAMES, p):
        print(f"  P({name}) = {prob:.4f}")
    return p


test_height = 68.0   # The value is subjected to change
_ = classify(test_height, mu, var, pi)

Height = 68.0 inches
  P(Shorter-height component) = 0.1988
  P(Taller-height component) = 0.8012


## 5. Why can splitting at the global mean be inappropriate?

Splitting all observations at the global mean and averaging each group is a simple approach, but it has some real limitations.

1. Hard assignments ignore uncertainty. Every observation goes into exactly one group, even if its height is right next to the cutoff. Because the two height distributions overlap, some observations will end up in the wrong group.

2. The estimates depend on where the cutoff falls. The means and variances we calculate from the two groups depend entirely on which observations land on each side of the global mean. As a result, they may not reflect the true mixture components.

3. The cutoff ignores the mixture parameters. A single split at the global mean doesn’t account for the separate variances and mixing weights of the two Gaussian components. The right classification boundary depends on both components’ distributions and on their mixing weights.

The Expectation-Maximization (EM) algorithm avoids these problems by using soft assignments. In the E-step, each observation receives a probability of belonging to each component. In the M-step, those probabilities are used to update the component means, variances, and mixing weights. The process repeats until the log-likelihood changes by less than the chosen tolerance, or until the maximum number of iterations is reached.